In [ ]:
# import modules
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split 
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
from wordcloud import WordCloud

In [ ]:
# Create sample dataset
data = {
    "review": [
        "I absolutely loved this product!",
        "Fantastic quality, highly recommend.",
        "The item was exactly as described.",
        "Superb experience, will order again.",
        "Very happy with the purchase.",
        "Excellent customer service and fast delivery.",
        "This is the best purchase I've made.",
        "Great value for money, very satisfied.",
        "The product exceeded my expectations.",
        "Five stars, amazing quality!",
        "I really hated this product.",
        "Poor quality, not worth the price.",
        "The item arrived broken and damaged.",
        "Worst experience, never buying again.",
        "Very disappointed with the purchase.",
        "Customer service was unhelpful.",
        "This is the worst purchase I've made.",
        "Terrible value for money, not satisfied.",
        "The product was below my expectations.",
        "One star, terrible quality!"
    ],
    "sentiment": [
        "positive", "positive", "positive", "positive", "positive",
        "positive", "positive", "positive", "positive", "positive",
        "negative", "negative", "negative", "negative", "negative",
        "negative", "negative", "negative", "negative", "negative"
    ]
}
df = pd.DataFrame(data)

df.to_csv("reviews.csv", index=False)

In [ ]:
# Load dataset

df = pd.read_csv("reviews.csv")
df.head()

In [ ]:
# Preprocessing

def preprocess(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text.lower())
    filtered_tokens = [lemmatizer.lemmatize(i) for i in tokens if i.isalpha() and i not in stop_words]
    return ' '.join(filtered_tokens)

df['processed_text'] = df['review'].apply(preprocess)
print(df.head())

In [ ]:
# Tokenizing & Padding

tokenizer = Tokenizer(num_words=10000) 
tokenizer.fit_on_texts(df['processed_text']) 

X = tokenizer.texts_to_sequences(df['processed_text']) 
X_pad = pad_sequences(X, padding='post', maxlen=5)  
print(X_pad[15:])

In [ ]:
# Encoding

label_encoder = LabelEncoder() 
y = label_encoder.fit_transform(df['sentiment']) 
print(y[5:15])

In [ ]:
# Model

model = Sequential() 

model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=50),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dense(1, activation='softmax')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
# Train model

X_train, X_test, y_train, y_test = train_test_split(X_pad, y, test_size=0.2, random_state=42) 
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

In [ ]:
# Evaluate

loss, accuracy = model.evaluate(X_test, y_test) 
print(f"Test Loss: {loss}") 
print(f"Test Accuracy: {accuracy}") 

In [ ]:
# Plotting

plt.plot(history.history['accuracy']) 
plt.plot(history.history['val_accuracy']) 
plt.title('Model accuracy') 
plt.xlabel('Epochs') 
plt.ylabel('Accuracy') 
plt.legend(['Train', 'Test'], loc='upper left') 
plt.show() 

plt.plot(history.history['loss']) 
plt.plot(history.history['val_loss']) 
plt.title('Model loss') 
plt.xlabel('Epochs') 
plt.ylabel('Loss') 
plt.legend(['Train', 'Test'], loc='upper left') 
plt.show()

In [ ]:
# Wordcloud
all_text = " ".join(df["processed_text"])

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white",
    colormap="viridis"
).generate(all_text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()
